In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [31]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent


In [18]:
loader = PyPDFLoader("C:\\GenAI\\data\\medical_report.pdf")
docs =loader.load()

In [19]:
len(docs)

9

In [20]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splited_docs = splitter.split_documents(docs)


In [21]:
len(splited_docs)

26

In [22]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small" )

In [23]:
vector_store = InMemoryVectorStore.from_documents(documents = splited_docs, embedding = embeddings)

### agent =tools, llm, prompt

In [52]:
#@tool
def retriever_tool(query):
    """This tool can help you to retrieve relevant data of the PDF Documents, and these documents are splitted into small chunks, so you can get more relevant data. have details about medical reports."""
    print("tool called:", query)
    docs = vector_store.similarity_search(query=query, k=4)
    print(docs[0].page_content)
    context = ""
    for doc in docs:
        context = doc.page_content +"\n\n"
    return context

In [53]:
llm = ChatOpenAI(model="gpt-5")    

In [54]:
System_prompt = """ 
You are a helpful assistant that answers question using retrieved context.
Always use the 'retriever_tool' tool for question requiring external knowledge.
"""

In [55]:
agent = create_agent(
    model=llm, 
    tools=[retriever_tool], 
    system_prompt=System_prompt
)

In [56]:
query = "what is the name of patient, and what is the name of doctor"
response = agent.invoke({"messages":[{"role":"user", "content": query}]})

tool called: patient name and doctor name
MD, Pathology
Chief of Laboratory                            
Dr Lal PathLabs Ltd
Dr Kiran Bhargava Pathak
MD, Pathology
Chief of Laboratory                            
Dr Lal PathLabs Ltd
*474764803*
.
Page 7 of 9
tool called: "Patient Name" OR Patient Name OR Name of Patient OR Patient: OR Name:
Report Status    
Female
27 Years:
:
:
:
Age
Gender
Reported        
P
9/7/2025   4:56:00PM
DR NITIN NAHAR
474764803
Ms. NIKITA  CHUDHARY:
:
:
:
:
Name        
Lab No.    
Ref By 
Collected       
A/c Status Final
10/7/2025  6:31:50PM
:Collected at            :Processed at             BHOPAL CC-82
Mr Rachel V John Pata So Vitus John Mig 26 
Graund,Indrapuri, Phone: 8770817968
 
LPL-NATIONAL REFERENCE LAB
National Reference laboratory, Block E, 
Sector 18, Rohini, New Delhi -110085
Test Report      
Test Name Results Units Bio. Ref. Interval
 ------------------------------------------------------------
Dr Beena Chandrasekhar
PhD, Life Sciences
Technica

In [59]:
result = response["messages"][-1].content

In [60]:
print(result)

- Patient name: Ms. Nikita Choudhary
- Doctor name: Dr. Nitin Nahar
